<a href="https://colab.research.google.com/github/Neeti1987/Pathway-GEM-scripts/blob/main/Threonine_pathway_specific_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === STEP 1: Install dependencies ===
!pip install cobra pandas

import cobra
from cobra import Model, Reaction, Metabolite
import pandas as pd
from google.colab import files

# === STEP 2: Upload the combined KO annotation file ===
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# === STEP 3: Parse KO annotations genome-wise ===
genome_kos = {}
with open(filename, "r") as f:
    for line in f:
        if not line.strip():
            continue
        parts = line.strip().split("\t")
        if len(parts) < 2:
            continue
        genome_id, ko = parts[0], parts[1]
        # genome prefix before first underscore (ADCAI, AFCAI, BTB, BTQ, etc.)
        genome_prefix = genome_id.split("_")[0]
        genome_kos.setdefault(genome_prefix, set()).add(ko)

# === STEP 4: Define KO logic for M00018 ===
thr_module_logic = {
    "Aspartokinase": [{"K00928","K12524","K12525","K12526"}],
    "Aspartate_semialdehyde_dehydrogenase": [{"K00133"}],
    "Homoserine_dehydrogenase": [{"K00003","K12524","K12525"}],
    "Homoserine_kinase": [{"K00872","K02204","K02203"}],
    "Threonine_synthase": [{"K01733"}]
}

def check_thr_module(ko_list):
    status = {}
    for enzyme, kosets in thr_module_logic.items():
        found = set()
        present = False
        for kos in kosets:
            if kos.intersection(ko_list):
                present = True
                found = kos.intersection(ko_list)
                break
        status[enzyme] = (present, found)
    complete = all([present for present, found in status.values()])
    return complete, status

# === STEP 5: Build KEGG-accurate threonine pathway model ===
def build_thr_model():
    model = Model("Threonine_pathway")

    # Metabolites
    asp = Metabolite("C00049_asp_c", compartment="c")
    atp = Metabolite("C00002_atp_c", compartment="c")
    adp = Metabolite("C00008_adp_c", compartment="c")
    asa_phosphate = Metabolite("C03082_asa_phosphate_c", compartment="c")
    asa_semialdehyde = Metabolite("C00441_asa_semialdehyde_c", compartment="c")
    nadp = Metabolite("C00006_nadp_c", compartment="c")
    nadph = Metabolite("C00005_nadph_c", compartment="c")
    nad = Metabolite("C00003_nad_c", compartment="c")
    nadh = Metabolite("C00004_nadh_c", compartment="c")
    h = Metabolite("C00080_h_c", compartment="c")
    h2o = Metabolite("C00001_h2o_c", compartment="c")
    pi = Metabolite("C00009_pi_c", compartment="c")
    hser = Metabolite("C00263_hser_c", compartment="c")
    phser = Metabolite("C01102_phser_c", compartment="c")
    thr = Metabolite("C00188_thr_c", compartment="c")

    # Reactions (KEGG equations)
    r00480 = Reaction("R00480_Aspartokinase")
    r00480.add_metabolites({asp:-1, atp:-1, adp:1, asa_phosphate:1})

    r02291 = Reaction("R02291_Aspartate_semialdehyde_dehydrogenase")
    r02291.add_metabolites({asa_semialdehyde:-1, pi:-1, nadp:-1,
                            asa_phosphate:1, nadph:1, h:1})

    r01773 = Reaction("R01773_Homoserine_dehydrogenase_NAD")
    r01773.lower_bound = -1000; r01773.upper_bound = 1000
    r01773.add_metabolites({hser:-1, nad:-1, asa_semialdehyde:1, nadh:1, h:1})

    r01775 = Reaction("R01775_Homoserine_dehydrogenase_NADP")
    r01775.lower_bound = -1000; r01775.upper_bound = 1000
    r01775.add_metabolites({hser:-1, nadp:-1, asa_semialdehyde:1, nadph:1, h:1})

    r01771 = Reaction("R01771_Homoserine_kinase")
    r01771.add_metabolites({hser:-1, atp:-1, adp:1, phser:1})

    r01466 = Reaction("R01466_Threonine_synthase")
    r01466.add_metabolites({phser:-1, h2o:-1, thr:1, pi:1})

    model.add_reactions([r00480, r02291, r01773, r01775, r01771, r01466])

    # Exchanges for all metabolites
    for met in [asp, atp, adp, asa_phosphate, asa_semialdehyde,
                nadp, nadph, nad, nadh, h, h2o, pi, phser, thr]:
        ex = Reaction(f"EX_{met.id}")
        ex.add_metabolites({met:1})
        ex.lower_bound = -1000
        ex.upper_bound = 1000
        model.add_reactions([ex])

    # Demand reaction for threonine
    dm_thr = Reaction("DM_C00188_thr_c")
    dm_thr.add_metabolites({thr:-1})
    dm_thr.lower_bound = 0
    dm_thr.upper_bound = 1000
    model.add_reactions([dm_thr])
    model.objective = dm_thr

    return model

# === STEP 6: Run analysis genome-wise ===
summary = []
for genome, kos in genome_kos.items():
    complete, status = check_thr_module(kos)
    flux_value = None
    if complete:
        model = build_thr_model()
        sol = model.optimize()
        flux_value = sol.objective_value
    summary.append({
        "Genome": genome,
        "Aspartokinase": status["Aspartokinase"][1],
        "Aspartate_semialdehyde_dehydrogenase": status["Aspartate_semialdehyde_dehydrogenase"][1],
        "Homoserine_dehydrogenase": status["Homoserine_dehydrogenase"][1],
        "Homoserine_kinase": status["Homoserine_kinase"][1],
        "Threonine_synthase": status["Threonine_synthase"][1],
        "Module_complete": complete,
        "Flux_to_threonine": flux_value
    })

# === STEP 7: Output genome-wise table ===
df = pd.DataFrame(summary)
print(df)
df.to_csv("threonine_flux_genomewise.csv", index=False)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.8/141.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 22.0 MB/s eta 0:00:00


Saving FINAL BLAST.tsv to FINAL BLAST.tsv
    Genome Aspartokinase Aspartate_semialdehyde_dehydrogenase  \
0    ADCAI      {K00928}                             {K00133}   
1    AFCAI      {K00928}                             {K00133}   
2      BTB      {K00928}                             {K00133}   
3      BTQ      {K00928}                             {K00133}   
4   BTQVLC      {K00928}                             {K00133}   
5     BTZ1      {K00928}                             {K00133}   
6    China      {K00928}                             {K00133}   
7   China1      {K00928}                             {K00133}   
8    MEAM1      {K00928}                             {K00133}   
9     PeMo      {K00928}                             {K00133}   
10    SiSi      {K00928}                             {K00133}   
11      TV      {K00928}                             {K00133}   

   Homoserine_dehydrogenase Homoserine_kinase Threonine_synthase  \
0                  {K00003}          {K02204

In [ ]:
# === Save results as TSV ===
# Assuming your results are stored in a DataFrame called df

# Export to TSV
df.to_csv("threonine_flux_genomewise.tsv", sep="\t", index=False)

# Download the TSV file
from google.colab import files
files.download("threonine_flux_genomewise.tsv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>